# **Australian Solar Generation Forecasting (30-min intervals)**

## **Step 1: Environment Setup & Time Series Fundamentals**

**Research/Learn:**

*   **Time series forecasting** predicts future values using historical data, crucial for trend and seasonal analysis. **Univariate forecasting** uses a **single** variable's past to predict its future, while **multivariate** uses **multiple** interdependent variables. **Probabilistic forecasts** provide a **probability distribution** of potential future values, capturing uncertainty better than deterministic point forecasts.

*   **NASA POWER** (Source API docs: https://power.larc.nasa.gov/api/pages/#/) provides analysis-ready solar and meteorological time series for
energy applications. The Temporal API family includes **Hourly**, **Daily**, **Monthly**, and **Climatology** services. Of these, Hourly is the most useful for our energy optimization engine because it provides hour-by-hour values, while Daily is a strong fallback or simpler first integration. NASA’s Temporal overview also shows that Hourly supports Point only, while Daily, Monthly, and Climatology support both Point and Regional. For our system, NASA POWER is useful because it adds a real environmental layer
for:
    * solar generation modeling
    * battery charge/discharge simulation
    * EV charging optimization
    * TOU-aware cost simulation
    * stronger scenario confidence

*   **Australian postcode datasets** are geographical collections that link postcodes across Australia to their corresponding physical locations. These datasets are essential for spatial analysis and localized data retrieval in technical projects.

    * **Core Attributes:** Each record typically includes the 4-digit Australian postcode, suburb/locality names, state, and precise geographic coordinates (Latitude and Longitude).

    * **Data Formats:** These datasets are commonly available in open-standard formats such as CSV for tabular processing or GeoJSON for mapping and spatial queries.

In [95]:
# install libraries ( transformers , datasets , gluonts , chronos , pandas , requests )
!pip install transformers datasets gluonts chronos pandas requests

In [96]:
# HF account
from huggingface_hub import HfApi

api = HfApi()

user_info = api.whoami()

user_info

{'type': 'user',
 'id': '68810e5b766eae4f6f2ac465',
 'name': 'codenhenhe',
 'fullname': 'Nguyễn Nhựt Linh',
 'email': 'linhnguyennhut07103@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1780272000,
 'isPro': False,
 'avatarUrl': '/avatars/029b932f62a41833fc86c6dac1282cda.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'solar-generation-forecasting',
   'role': 'write',
   'createdAt': '2026-05-06T03:31:37.355Z'}}}

In [97]:
# HF token
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

print(type(HF_TOKEN))

<class 'str'>


In [98]:
!curl --location 'https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=ALLSKY_SFC_SW_DWN%2CT2M&community=RE&longitude=151.21&latitude=-33.86&start=20200101&end=20200131&format=JSON'

{"type":"Feature","geometry":{"type":"Point","coordinates":[151.21,-33.86,31.72]},"properties":{"parameter":{"ALLSKY_SFC_SW_DWN":{"2020010100":0.0,"2020010101":0.0,"2020010102":0.0,"2020010103":0.0,"2020010104":0.0,"2020010105":45.55,"2020010106":174.15,"2020010107":300.5,"2020010108":527.15,"2020010109":654.8,"2020010110":904.75,"2020010111":985.45,"2020010112":994.53,"2020010113":906.45,"2020010114":769.58,"2020010115":493.6,"2020010116":316.42,"2020010117":162.82,"2020010118":27.58,"2020010119":0.0,"2020010120":0.0,"2020010121":0.0,"2020010122":0.0,"2020010123":0.0,"2020010200":0.0,"2020010201":0.0,"2020010202":0.0,"2020010203":0.0,"2020010204":0.0,"2020010205":28.65,"2020010206":96.5,"2020010207":176.4,"2020010208":272.85,"2020010209":416.5,"2020010210":548.33,"2020010211":564.88,"2020010212":606.42,"2020010213":587.5,"2020010214":480.67,"2020010215":385.58,"2020010216":263.38,"2020010217":105.2,"2020010218":19.1,"2020010219":0.0,"2020010220":0.0,"2020010221":0.0,"2020010222":0.0,"

In [99]:
import requests
import pandas as pd

def get_coordinates(postcode):
    lookup_table = {
        "2000": {"lat": -33.86, "long": 151.21},
    }
    return lookup_table.get(postcode, None)

def fetch_nasa_data(lat, long, start_date="20200101", end_date="20200131"):
    url = f"https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=ALLSKY_SFC_SW_DWN%2CT2M&community=RE&longitude={long}&latitude={lat}&start={start_date}&end={end_date}&format=JSON"

    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        df = pd.DataFrame(data['properties']['parameter'])
        return df
    else:
        return f"Error: {response.status_code}"

In [100]:
postcode_test = "2000"
coords = get_coordinates(postcode_test)

if coords:
    print(f"Fetching data for Postcode {postcode_test} (Lat: {coords['lat']}, Lon: {coords['long']})...")
    result_df = fetch_nasa_data(coords['lat'], coords['long'])
    print(result_df)
else:
    print("Postcode không tồn tại trong hệ thống test.")

Fetching data for Postcode 2000 (Lat: -33.86, Lon: 151.21)...
            ALLSKY_SFC_SW_DWN    T2M
2020010100                0.0  20.19
2020010101                0.0  19.68
2020010102                0.0  19.38
2020010103                0.0  19.20
2020010104                0.0  19.19
...                       ...    ...
2020013119                0.0  25.12
2020013120                0.0  24.83
2020013121                0.0  24.65
2020013122                0.0  24.52
2020013123                0.0  24.34

[744 rows x 2 columns]


## **Step 2: Postcode to Lat/Long Mapping**

Link dataset: https://github.com/Elkfox/Australian-Postcode-Data

In [101]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [102]:
df_origin = pd.read_csv("/content/drive/MyDrive/TT/au_postcodes.csv")

In [103]:
df_origin

,postcode,place_name,state_name,state_code,latitude,longitude,accuracy
0,200,Australian National University,Australian Capital Territory,ACT,-35.2777,149.1189,1.0
1,221,Barton,Australian Capital Territory,ACT,-35.3049,149.1412,4.0
2,2540,Wreck Bay,Australian Capital Territory,ACT,-35.1627,150.6907,4.0
3,2540,Hmas Creswell,Australian Capital Territory,ACT,-35.0280,150.5501,3.0
4,2540,Jervis Bay,Australian Capital Territory,ACT,-35.1333,150.7000,4.0
...,...,...,...,...,...,...,...
16870,6989,Maddington,Western Australia,WA,-32.0500,115.9833,4.0
16871,6990,Gosnells,Western Australia,WA,-32.0810,116.0054,4.0
16872,6991,Kelmscott,Western Australia,WA,-32.1243,116.0259,4.0
16873,6992,Armadale,Western Australia,WA,-32.1461,116.0093,4.0


In [104]:
df_main = df_origin.groupby('postcode')[['latitude', 'longitude']].mean().reset_index()

In [105]:
def lat_long_mapping(postcode):
  res = df_main.loc[df_main['postcode'] == postcode, ['latitude', 'longitude']]
  if not res.empty:
    return res['latitude'].mean(), res['longitude'].mean()
  return None, None

In [106]:
lat, long = lat_long_mapping(3000)

print(f"lat: {lat}")
print(f"long: {long}")

lat: -37.814
long: 144.9633


## **Step 3: NASA POWER API Integration**

In [107]:
postcode = 3000
lat, long = lat_long_mapping(postcode)

In [127]:
import requests
import json
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta

def fetch_and_save_nasa_data(postcode, lat, long, start_date=None, end_date=None, save_dir="data/raw"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        print(f"Đã tạo thư mục: {save_dir}")

    if start_date is None or end_date is None:
        now = datetime.now()
        dt_end = now - relativedelta(days=1)
        dt_start = dt_end - relativedelta(months=24)

        if end_date is None:
            end_date = dt_end.strftime("%Y%m%d")
        if start_date is None:
            start_date = dt_start.strftime("%Y%m%d")

    url = (
        f"https://power.larc.nasa.gov/api/temporal/hourly/point?"
        f"parameters=T2M,ALLSKY_SFC_SW_DWN,WS10M&community=RE&longitude={long}&latitude={lat}"
        f"&start={start_date}&end={end_date}&format=JSON"
    )

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        file_path = os.path.join(save_dir, f"postcode_{postcode}.json")

        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

        print(f"Thành công! Đã lưu tại: {file_path}")
        return data

    except requests.exceptions.RequestException as e:
        error_msg = f"Lỗi kết nối API cho postcode {postcode}: {e}"
        print(error_msg)
        return None

In [131]:
data_auto = fetch_and_save_nasa_data(postcode, lat, long, save_dir="/content/drive/MyDrive/TT/data/raw")
data_auto

Đã tạo thư mục: /content/drive/MyDrive/TT/data/raw
Thành công! Đã lưu tại: /content/drive/MyDrive/TT/data/raw/postcode_3000.json


{'type': 'Feature',
 'geometry': {'type': 'Point', 'coordinates': [144.963, -37.814, 58.74]},
 'properties': {'parameter': {'T2M': {'2024050500': 10.97,
    '2024050501': 10.79,
    '2024050502': 10.73,
    '2024050503': 10.66,
    '2024050504': 10.49,
    '2024050505': 10.4,
    '2024050506': 10.3,
    '2024050507': 10.48,
    '2024050508': 12.26,
    '2024050509': 13.78,
    '2024050510': 15.27,
    '2024050511': 16.46,
    '2024050512': 17.25,
    '2024050513': 17.81,
    '2024050514': 17.98,
    '2024050515': 17.74,
    '2024050516': 16.91,
    '2024050517': 14.9,
    '2024050518': 14.0,
    '2024050519': 13.66,
    '2024050520': 13.25,
    '2024050521': 12.73,
    '2024050522': 12.12,
    '2024050523': 11.59,
    '2024050600': 11.12,
    '2024050601': 10.74,
    '2024050602': 10.43,
    '2024050603': 10.18,
    '2024050604': 10.04,
    '2024050605': 9.88,
    '2024050606': 9.74,
    '2024050607': 10.1,
    '2024050608': 12.69,
    '2024050609': 14.58,
    '2024050610': 15.97,
    

## Step 4: Feature Engineering & Solar Generation Calculation

### 1. Solar Power Output Formula ($P$)
(Documentation Link: https://docs.nlr.gov/docs/fy14osti/62641.pdf)

This formula is based on the NREL PVWatts v5 model, which is the industry standard for estimating the electricity production of photovoltaic (PV) systems.  $$P = C \times \frac{G}{1000} \times [1 + \gamma \times (T_{cell} - 25)] \times PR$$Parameters:
* $P$: Power output ($kWh$ for the given time interval).  
* $C$: Nameplate DC capacity of the system ($kWp$).  
* $G$: Incident solar radiation from NASA ($W/m^2$) — variable ALLSKY_SFC_SW_DWN.  
* $1000$: Reference irradiance at Standard Test Conditions (STC) ($W/m^2$).  
* $\gamma$: Temperature coefficient (typically $-0.004$ or $-0.4\%$ per $^\circ C$ for crystalline silicon panels).  
* $T_{cell}$: The actual temperature of the solar cells ($^\circ C$).  
* $25$: Reference cell temperature at STC ($^\circ C$).  
* $PR$: Performance Ratio (typically $0.75$ to $0.80$), accounting for system losses like inverter efficiency, wiring, and soiling.  

### 2. Cell Temperature Estimation Formula ($T_{cell}$)
Since NASA provides ambient air temperature ($T_{ambient}$), we use a simplified version of the Sandia National Laboratories model to estimate the cell temperature:  
$$T_{cell} = T_{ambient} + (G \times 0.025)$$

Parameters:
* $T_{ambient}$: Ambient air temperature at 2 meters from NASA
* ($^\circ C$) — variable T2M.  
* $0.025$: The heat absorption coefficient (Ross coefficient) for roof-mounted systems.  

### 3. Energy Calculation for 30-Minute Intervals
When performing Linear Interpolation to convert hourly NASA data into 30-minute intervals, the energy ($kWh$) generated in that 30-minute window must be adjusted because the original formula calculates power for a full hour:  
$$P_{30min} = \frac{P_{formula}}{2}$$
(Or multiply by $0.5$ hours to convert the instantaneous power into energy for that specific time step).  

In [138]:
import pandas as pd
import json

def process_step_4(json_data, capacity_kwp=1.0):

    params = json_data['properties']['parameter']

    df = pd.DataFrame(params)

    df.index = pd.to_datetime(df.index, format='%Y%m%d%H')
    df.index.name = 'timestamp'

    df_30min = df.resample('30min').interpolate(method='linear')

    pr = 0.77            # Hệ số hiệu suất hệ thống (tổn thất ~23%)
    temp_coeff = -0.004  # Hệ số nhiệt độ (giảm 0.4% mỗi độ trên 25°C)

    df_30min['T_cell'] = df_30min['T2M'] + (df_30min['ALLSKY_SFC_SW_DWN'] * 0.025)

    df_30min['temp_factor'] = 1 + temp_coeff * (df_30min['T_cell'] - 25)
    df_30min['temp_factor'] = df_30min['temp_factor'].clip(upper=1.0)

    # Công thức tính sản lượng điện (kWh) cho mỗi 30 phút
    df_30min['solar_gen_kwh'] = (capacity_kwp *
                                (df_30min['ALLSKY_SFC_SW_DWN'] / 1000) *
                                df_30min['temp_factor'] *
                                pr) / 2

    df_30min['solar_gen_kwh'] = df_30min['solar_gen_kwh'].clip(lower=0)

    df_30min['hour'] = df_30min.index.hour
    df_30min['day_of_year'] = df_30min.index.dayofyear
    df_30min['month'] = df_30min.index.month

    return df_30min

# data_auto = json.loads(your_json_string)

processed_df = process_step_4(data_auto, capacity_kwp=5.0)

print(processed_df[['ALLSKY_SFC_SW_DWN', 'T2M', 'solar_gen_kwh']][:20])

# processed_df.to_csv("solar_training_data_30min.csv")

                     ALLSKY_SFC_SW_DWN     T2M  solar_gen_kwh
timestamp                                                    
2024-05-05 00:00:00              0.000  10.970       0.000000
2024-05-05 00:30:00              0.000  10.880       0.000000
2024-05-05 01:00:00              0.000  10.790       0.000000
2024-05-05 01:30:00              0.000  10.760       0.000000
2024-05-05 02:00:00              0.000  10.730       0.000000
2024-05-05 02:30:00              0.000  10.695       0.000000
2024-05-05 03:00:00              0.000  10.660       0.000000
2024-05-05 03:30:00              0.000  10.575       0.000000
2024-05-05 04:00:00              0.000  10.490       0.000000
2024-05-05 04:30:00              0.000  10.445       0.000000
2024-05-05 05:00:00              0.000  10.400       0.000000
2024-05-05 05:30:00              0.000  10.350       0.000000
2024-05-05 06:00:00              0.000  10.300       0.000000
2024-05-05 06:30:00             11.150  10.390       0.021464
2024-05-